In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import datetime as dt
import os
import shutil
import subprocess
import time
import Metashape
import requests

In [0]:
run_date = dt.datetime.now().strftime("%Y-%m-%d")

# This is the Batch File from Jorge. (click to view)
<?xml version="1.0" encoding="UTF-8"?>
<batchjobs version="2.1.1">
  <job name="AlignPhotos" enabled="false" target="all">
    <keypoint_limit>60000</keypoint_limit>
    <keypoint_limit_per_mpx>4000</keypoint_limit_per_mpx>
    <mask_tiepoints>false</mask_tiepoints>
    <tiepoint_limit>0</tiepoint_limit>
  </job>
  <job name="OptimizeCameras" enabled="false" target="all">
    <fit_b1>true</fit_b1>
    <fit_b2>true</fit_b2>
    <fit_k4>true</fit_k4>
  </job>
  <job name="BuildPointCloud" enabled="false" target="all">
    <downscale>2</downscale>
    <filter_mode>3</filter_mode>
    <reuse_depth>true</reuse_depth>
  </job>
  <job name="DetectMarkers" enabled="false" target="all">
    <tolerance>80</tolerance>
  </job>
  <job name="LocateReflectancePanels" enabled="false" target="all"/>
  <job name="CalibrateReflectance" enabled="false" target="all">
    <use_sun_sensor>true</use_sun_sensor>
  </job>
  <job name="BuildDem" enabled="false" target="all">
    <downscale>2</downscale>
    <filter_mode>3</filter_mode>
    <reuse_depth>true</reuse_depth>
  </job>
  <job name="BuildOrthomosaic" enabled="false" target="all"/>
</batchjobs>

In [0]:

# # Step 1: Start Metashape in offscreen mode (non-blocking)
# metashape_process = subprocess.Popen(
#     ["/opt/agisoft/metashape-pro/metashape.sh", "-platform", "offscreen"],
#     stdout=subprocess.DEVNULL,
#     stderr=subprocess.DEVNULL
# )

# time.sleep(15)  # Wait until metsashape is ready

print(Metashape.app.version)

 Agisoft License Key: ([REDACTED])


In [0]:

# ── Read the list of flights to process ────────────────────────────────────
# This notebook is triggered with a parameter called "flight_metadata_paths",
# which contains one or more flight JSON paths separated by commas.
# This is how the orchestrator tells this task WHICH flights to process.
dbutils.widgets.text("flight_metadata_paths", "")
raw_paths = dbutils.widgets.get("flight_metadata_paths")

# If no paths were passed in, there's nothing to do — stop the notebook early.
if not raw_paths:
    dbutils.notebook.exit("Error: No flight path was received from the trigger. Shutting down node.")

# Split the comma-separated string into a list of individual flight paths.
flights_list = raw_paths.split(',')

# Convert Databricks-style paths ("dbfs:/...") into local filesystem-style
# paths ("/dbfs/...") so they can be used with normal Python file operations.
flights_list = [p.replace("dbfs:/", "/dbfs/") for p in flights_list]

print(f" Having received the order to process a block of {len(flights_list)} flights.")

# Configure Metashape to use the GPU for processing and disable CPU-only mode
# (this speeds up the heavy photogrammetry steps below).
Metashape.app.gpu_mask = 1
Metashape.app.cpu_enable = False

# ── Process each flight one at a time ───────────────────────────────────────
for json_path in flights_list:
    print("\n" + "="*70)
    print(f" STARTING MISSION: {json_path}")
    print("="*70)

    # The flight's project folder is simply the folder containing its metadata file.
    PROJECT_DIR = os.path.dirname(json_path)

    # Final results (report, DEM, orthomosaic, etc.) will be uploaded back here.
    OUTPUTS_DIR = PROJECT_DIR

    # ── Detect which type of imagery this flight has ───────────────────────
    # Flights can contain either multispectral ("multi-spec") or RGB ("rgb")
    # images. We check which folder exists and set the sensor type accordingly.
    base_raw_dir = os.path.join(PROJECT_DIR, "raw_data")
    sensor_type = ""
    if os.path.exists(os.path.join(base_raw_dir, "multi-spec")):
        RAW_IMG_DIR = os.path.join(base_raw_dir, "multi-spec")
        sensor_type = "MS"
        print("  Image directory detected: multi-spec")
    elif os.path.exists(os.path.join(base_raw_dir, "rgb")):
        RAW_IMG_DIR = os.path.join(base_raw_dir, "rgb")
        sensor_type = "RGB"
        print("  Image directory detected: rgb")
    else:
        # If neither folder exists, we can't process this flight — skip it
        # and move on to the next one in the list.
        print(f"  Error: The 'rgb' and 'multi-spec' subfolders could not be found in {base_raw_dir}")
        continue

    # ── Set up local (SSD) working folders ──────────────────────────────────
    # Metashape performs much better reading/writing from local disk rather
    # than directly from DBFS/cloud storage, so we copy images to a temporary
    # local folder before processing, and copy results back at the end.
    flight_name = os.path.basename(PROJECT_DIR)
    local_tmp = f"/tmp/metashape_{flight_name}"
    input_images_local = f"{local_tmp}/local_images"
    outputs_local = f"{local_tmp}/outputs"

    # Processing options for this flight:
    GCP = False                       # Whether Ground Control Points / markers should be used
    has_reflectance_panels = True     # Whether reflectance calibration panels are present in the images

    # Wipe any leftover temp folder from a previous run, then recreate clean folders.
    if os.path.exists(local_tmp):
        shutil.rmtree(local_tmp)

    os.makedirs(input_images_local, exist_ok=True)
    os.makedirs(outputs_local, exist_ok=True)

    # Copy the raw images from cloud storage into the local temp folder.
    if os.path.exists(RAW_IMG_DIR):
        pictures = [f for f in os.listdir(RAW_IMG_DIR) if f.lower().endswith(('.tif', '.jpg', '.jpeg'))]
        print(f" Copy {len(pictures)} images into local SSD...")
        for img in pictures:
            shutil.copy2(os.path.join(RAW_IMG_DIR, img), os.path.join(input_images_local, img))
    else:
        # No raw_data folder for this flight — skip it and continue with the next one.
        print(f" Error: There is no Raw_Data folder in {PROJECT_DIR}. skipping flight.")
        continue

    # Build the final list of local image paths that will be fed into Metashape.
    photos = [os.path.join(input_images_local, f) for f in os.listdir(input_images_local) if f.lower().endswith(('.tif', '.jpg', '.jpeg'))]
    print(f" Starting metashape with {len(photos)} images...")

    try:
        # ── Activate the Agisoft Metashape license ──────────────────────────
        # The license key is stored securely in a Databricks secret scope,
        # not hardcoded here.
        agisoft_license_key = dbutils.secrets.get(scope="agisoft_creds", key="agisoft_license_key")
        Metashape.license.activate(agisoft_license_key)

        # Create a new Metashape project ("chunk") for this flight.
        doc = Metashape.Document()
        doc.save(local_tmp + '/project.psx')
        chunk = doc.addChunk()

        print('Adding Markers if there are any')
        # If Ground Control Points are enabled, add each marker's known
        # real-world coordinates to the project so Metashape can use them
        # to improve georeferencing accuracy.
        if GCP:
            if 'marker_gdf' in locals() and len(marker_gdf) > 0:
                for idx, row in marker_gdf.iterrows():
                    marker = chunk.addMarker()
                    marker.label = row['label']
                    marker.reference.location = Metashape.Vector([row.x, row.y, row.h])
                    marker.reference.enabled = True

        # ── Core Metashape photogrammetry pipeline ──────────────────────────
        print('Adding the photos')
        chunk.addPhotos(photos)
        doc.save()

        # Detect matching features across photos (this is what lets Metashape
        # figure out how the images overlap with each other).
        print('Match the photos')
        chunk.matchPhotos(keypoint_limit = 60000,
                          tiepoint_limit = 0,
                          keypoint_limit_per_mpx = 4000,
                          generic_preselection = True,
                          reference_preselection = True)
        doc.save()

        # Estimate each camera's position and orientation based on the matched photos.
        print('Align the Cameras')
        chunk.alignCameras()
        doc.save()
        chunk.updateTransform()

        # Refine the camera calibration parameters for better accuracy.
        print('Optimize the cameras')
        chunk.optimizeCameras(fit_b1=True, fit_b2=True, fit_k4=True)
        doc.save()

        # Generate depth maps, which are used to build the dense 3D point cloud.
        print('Build the Depth Maps')
        chunk.buildDepthMaps(downscale = 2, filter_mode = Metashape.NoFiltering, max_neighbors = 16)
        doc.save()

        # If GCPs are enabled, automatically detect coded targets in the images
        # and import their reference coordinates.
        print('Detect the markers and import the reference points')
        if GCP:
            chunk.detectMarkers(target_type=Metashape.TargetType.CircularTarget, tolerance=80, filter_mask=False, maximum_residual=15)
            chunk.importReference()
        else:
            print('No Marker')

        # Locate the reflectance calibration panels in the images (needed to
        # correct the multispectral/RGB values to true reflectance values).
        print('Locate the Reflectance Panels')
        if has_reflectance_panels:
            chunk.locateReflectancePanels()
            doc.save()

        # Use the located panels to calibrate reflectance across all images.
        print('Calibrate the Reflectance Panels')
        chunk.calibrateReflectance()
        doc.save()

        # Build the dense 3D point cloud from the depth maps.
        print('Build Point Cloud')
        chunk.buildPointCloud()
        doc.save()

        # Build a Digital Elevation Model (DEM) from the point cloud.
        print('Build DEM')
        chunk.buildDem(source_data=Metashape.PointCloudData)
        doc.save()

        # Build the final orthomosaic (the stitched, georeferenced top-down image)
        # using the DEM as the surface reference.
        print('Build Orthomosaic')
        chunk.buildOrthomosaic(surface_data=Metashape.ElevationData)
        doc.save()

        # ── Export the results to the local temp folder ─────────────────────
        print('Exporting results to local temporary folder...')
        chunk.exportReport(outputs_local + '/report.pdf')

        if chunk.model:
            chunk.exportModel(outputs_local + '/model.obj')

        if chunk.elevation:
            chunk.exportRaster(outputs_local + '/DEM.tif', source_data = Metashape.ElevationData)

        if chunk.orthomosaic:
            # Name the exported orthomosaic file according to its sensor type
            # (multispectral vs. standard RGB).
            if sensor_type =="MS":
                chunk.exportRaster(outputs_local + '/MS.tif', source_data = Metashape.OrthomosaicData)
                print("Multispectral orthomosaic exported as MS.tif")
            else:
                chunk.exportRaster(outputs_local + '/RGB.tif', source_data = Metashape.OrthomosaicData)
                print("Standard orthomosaic exported as RGB.tif")

        print(f'Processing finished, results saved to {outputs_local}.')

        # ── Upload the final results back to cloud storage ──────────────────
        # Copy everything generated locally back into the flight's project
        # folder, so downstream jobs/tasks can pick them up.
        print(f'Uploading final results to the flight folder: {OUTPUTS_DIR}')
        generated_files = os.listdir(outputs_local)
        for file in generated_files:
            origen = os.path.join(outputs_local, file)
            destino = os.path.join(OUTPUTS_DIR, file)
            shutil.copy2(origen, destino)

        print(f' Mission {flight_name} Completed and files successfully uploaded.')

    except Exception as e:
        # If anything fails during processing, log the error and move on to
        # the next flight rather than stopping the whole batch.
        print(f" An error occurred in the processing of {flight_name}: {e}")

    finally:
        # Always deactivate the Metashape license and clean up the local temp
        # folder, whether the flight succeeded or failed, to free up disk
        # space and avoid license conflicts with other running tasks.
        Metashape.license.deactivate()

        if os.path.exists(local_tmp):
            shutil.rmtree(local_tmp)
            print(f" Cleaning the local SSD environment ({local_tmp}) completed.")

print("\n The entire batch of pending missions has been processed.")

In [0]:

# ── Job ID to trigger ────────────────────────────────────────────────────
# This is the Databricks Job ID of "plot_clipping" (Job 3). Once this
# notebook's own work is done, it will kick off that job so the heavier
# processing pipeline can run next.
JOB_3_ID = 162763862056473  #change with the plot_clipping job ID

# ── Get the current workspace context (host + auth token) ───────────────
# This lets the notebook call the Databricks REST API on its own, using
# the same workspace/credentials it's currently running in — no need to
# hardcode a host URL or token.
ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
host = ctx.apiUrl().get()
token = ctx.apiToken().get()

# ── Build the API request to trigger the job ─────────────────────────────
# "run-now" starts an existing job immediately (equivalent to clicking
# "Run now" in the Databricks UI), using the job_id defined above.
url = f"{host}/api/2.1/jobs/run-now"
headers = {"Authorization": f"Bearer {token}"}
data = {"job_id": JOB_3_ID}

# ── Send the request and report the outcome ──────────────────────────────
response = requests.post(url, headers=headers, json=data)

if response.status_code == 200:
    # A 200 response means Databricks accepted the request and started the run.
    print(" Trigger successful! Heavy processing Job has been started.")
else:
    # Anything else means the trigger failed — print the API's error message
    # so it's clear what went wrong (e.g. wrong job ID, permissions issue).
    print(f" Error triggering Job: {response.text}")